# Tóm tắt bằng xếp hạng câu trên đồ thị

## Ý tưởng (Step by step):

- [x] Document
- [x] Tiền xử lý văn bản
- [ ] Tách câu 
- [ ] Chuẩn hóa và tách từ
- [x] Biểu diễn mỗi câu thành vector
- [ ] Tính độ tương đồng giữa các câu
- [ ] Xây dựng ma trận tương đồng
- [ ] Xây dựng đồ thị câu có trọng số
- [ ] Áp dụng PageRank / TextRank
- [ ] Xếp hạng câu theo score
- [ ] Chọn câu theo số lượng hoặc độ dài cho phép
- [ ] Sắp xếp lại theo thứ tự xuất hiện ban đầu
- [ ] Candidate Summary

In [3]:
# --------------------
# Document
# --
# Lấy các đoạn văn đã có trong tài liệu
# --------------------

from typing import Union
from pathlib import Path
from bs4 import BeautifulSoup

# Đọc file txt
file_txt_path = Path.cwd().parent / "data" / "DUC_TEXT" / "train" / "d061j"

try:
    read_txt = file_txt_path.read_text(encoding="utf-8")
except FileNotFoundError:
    print("File not found")
else:
    print(f"Your file length: {len(read_txt)} characters")

# Trích xuất văn bản từ file
soup = BeautifulSoup(read_txt, "html.parser")

txt_dict_s: list[dict[str, Union[str, int, list, None]]] = []

for tag in soup.find_all("s"):
    text_dict = {
        "docid": tag.get("docid"),
        "num": tag.get("num"),
        "wdcount": tag.get("wdcount"),
        "content": tag.get_text(separator="", strip=True)
    }
    txt_dict_s.append(text_dict)

print(f"Have {len(txt_dict_s)} paragraphs")

print(f"đoạn text {txt_dict_s}")

Your file length: 32448 characters
Have 186 paragraphs
đoạn text [{'docid': 'AP880911-0016', 'num': '9', 'wdcount': '28', 'content': 'Hurricane Gilbert swept toward the Dominican Republic Sunday, and the Civil Defense alerted its heavily populated south coast to prepare for high winds, heavy rains and high seas.'}, {'docid': 'AP880911-0016', 'num': '10', 'wdcount': '17', 'content': 'The storm was approaching from the southeast with sustained winds of 75 mph gusting to 92 mph.'}, {'docid': 'AP880911-0016', 'num': '11', 'wdcount': '20', 'content': "``There is no need for alarm,'' Civil Defense Director Eugenio Cabral said in a television alert shortly before midnight Saturday."}, {'docid': 'AP880911-0016', 'num': '12', 'wdcount': '13', 'content': "Cabral said residents of the province of Barahona should closely follow Gilbert's movement."}, {'docid': 'AP880911-0016', 'num': '13', 'wdcount': '22', 'content': 'An estimated 100,000 people live in the province, including 70,000 in the city o

In [2]:
import math
from collections import Counter

documents = [
    "HUTECH HUTECH HUTECH IRS CaoHoc",
    "HUTECH HUTECH HUTECH AI CaoHoc",
    "HUTECH HUTECH HUTECH OOP DaiHoc",
]

def tokenize(document: str) -> list[str]:
    """
    Tách văn bản thành DS các từ
    """
    return document.lower().split()

def calculate_tf(tokens: list[str]) -> dict[str, float]:
    """
    TF(t, d)
    """
    word_counts = Counter(tokens)
    total_words = len(tokens)

    tf = {}

    for word, count in word_counts.items():
        tf[word] = count / total_words

    return tf

def calculate_idf(documents_tokens: list[list[str]]) -> dict[str, float]:
    """
    IDF(t) = log(N / df(t))
    N     = tổng số document
    df(t) = số document chứa term t
    """

    total_documents = len(documents_tokens)
    document_frequency = Counter()

    for tokens in documents_tokens:
        unique_terms = set(tokens)
        for term in unique_terms:
            document_frequency[term] += 1

    idf = {}

    for term, frequency in document_frequency.items():
        idf[term] = math.log(total_documents / frequency)

    return idf

def calculate_tfidf(tf: dict[str, float], idf: dict[str, float]) -> dict[str, float]:
    """
    TF-IDF(t, d) = TF(t, d) * IDF(t)
    """
    tfidf = {}
    for term, tf_value in tf.items():
        tfidf[term] = tf_value * idf[term]
    return tfidf


documents_tokens = []

for document in documents:
    tokens = tokenize(document)
    print(f"token: {tokens}")
    documents_tokens.append(tokens)

print(f"documents_tokens: {documents_tokens}")
# 2. Tính IDF trên toàn corpus

idf = calculate_idf(documents_tokens)

# 3. Tính TF-IDF cho từng document

for index, tokens in enumerate(documents_tokens, start=1):
    tf = calculate_tf(tokens)
    tfidf = calculate_tfidf(tf, idf)

    print(f"\nDocument d{index}")

    for term, score in tfidf.items():
        print(
            f"{term:10} "
            f"TF={tf[term]:.4f} "
            f"IDF={idf[term]:.4f} "
            f"TF-IDF={score:.4f}"
        )

token: ['hutech', 'hutech', 'hutech', 'irs', 'caohoc']
token: ['hutech', 'hutech', 'hutech', 'ai', 'caohoc']
token: ['hutech', 'hutech', 'hutech', 'oop', 'daihoc']
documents_tokens: [['hutech', 'hutech', 'hutech', 'irs', 'caohoc'], ['hutech', 'hutech', 'hutech', 'ai', 'caohoc'], ['hutech', 'hutech', 'hutech', 'oop', 'daihoc']]

Document d1
hutech     TF=0.6000 IDF=0.0000 TF-IDF=0.0000
irs        TF=0.2000 IDF=1.0986 TF-IDF=0.2197
caohoc     TF=0.2000 IDF=0.4055 TF-IDF=0.0811

Document d2
hutech     TF=0.6000 IDF=0.0000 TF-IDF=0.0000
ai         TF=0.2000 IDF=1.0986 TF-IDF=0.2197
caohoc     TF=0.2000 IDF=0.4055 TF-IDF=0.0811

Document d3
hutech     TF=0.6000 IDF=0.0000 TF-IDF=0.0000
oop        TF=0.2000 IDF=1.0986 TF-IDF=0.2197
daihoc     TF=0.2000 IDF=1.0986 TF-IDF=0.2197


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [ ]:
# ------------
# Tính độ tương đồng giữa các câu
# ------------
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Group theo docid
grouped = {}

for txt_dict in txt_dict_s:
    docid = txt_dict["docid"]

    if docid not in grouped:
        grouped[docid] = []

    grouped[docid].append(txt_dict)


for docid, sentences in grouped.items():
    # Giữ thứ tự câu theo tài liệu gốc
    sentences = sorted(
        sentences,
        key=lambda sentence: sentence["num"]
    )

    # Chỉ giữ các câu có token hợp lệ
    valid_sentences = [
        sentence
        for sentence in sentences
        if isinstance(sentence.get("preprocessed"), list)
        and len(sentence["preprocessed"]) > 0
    ]

    corpus = [
        " ".join(sentence["preprocessed"])
        for sentence in valid_sentences
    ]

    if not corpus:
        print(f"docid={docid} không có câu hợp lệ")
        continue

    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(corpus)

    similarity_matrix = cosine_similarity(tfidf_matrix)

    # Không nối một câu với chính nó
    np.fill_diagonal(similarity_matrix, 0.0)

    print(
        f"docid={docid}"
        f" | số câu hợp lệ={len(valid_sentences)}"
        f" | TF-IDF={tfidf_matrix.shape}"
        f" | similarity={similarity_matrix.shape}"
    )

docid=AP880911-0016 | số câu hợp lệ=16 | TF-IDF=(16, 168) | similarity=(16, 16)
docid=AP880912-0095 | số câu hợp lệ=32 | TF-IDF=(32, 376) | similarity=(32, 32)
docid=AP880912-0137 | số câu hợp lệ=30 | TF-IDF=(30, 322) | similarity=(30, 30)
docid=AP880915-0003 | số câu hợp lệ=56 | TF-IDF=(56, 467) | similarity=(56, 56)
docid=AP880916-0060 | số câu hợp lệ=35 | TF-IDF=(35, 266) | similarity=(35, 35)
docid=WSJ880912-0064 | số câu hợp lệ=17 | TF-IDF=(17, 176) | similarity=(17, 17)


In [29]:
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

summaries = {}
TOP_N = 3

for docid, sentences in grouped.items():
    sentences = sorted(sentences, key=lambda s: s["num"])

    valid_sentences = [
        s for s in sentences
        if isinstance(s.get("preprocessed"), list)
        and len(s["preprocessed"]) > 0
    ]

    if len(valid_sentences) < 2:
        summaries[docid] = valid_sentences[0]["content"] if valid_sentences else ""
        continue

    corpus = [" ".join(s["preprocessed"]) for s in valid_sentences]

    # Vector hóa
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus)

    # Ma trận tương đồng
    similarity_matrix = cosine_similarity(tfidf_matrix)
    np.fill_diagonal(similarity_matrix, 0.0)

    # Đồ thị + PageRank
    graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(graph, alpha=0.85, weight="weight")

    # Xếp hạng
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # Chọn top N, giữ thứ tự gốc
    top_indices = sorted([i for i, _ in ranked[:TOP_N]])
    summary = " ".join(valid_sentences[i]["content"] for i in top_indices)
    summaries[docid] = summary

# In kết quả
for docid, summary in summaries.items():
    print(f"\n=== {docid} ===\n{summary}")


=== AP880911-0016 ===
The storm was approaching from the southeast with sustained winds of 75 mph gusting to 92 mph. The National Weather Service in San Juan, Puerto Rico, said Gilbert was moving westward at 15 mph with a ``broad area of cloudiness and heavy weather'' rotating around the center of the storm. On Saturday, Hurricane Florence was downgraded to a tropical storm and its remnants pushed inland from the U.S. Gulf Coast.

=== AP880912-0095 ===
``All interests in the Western Caribbean should continue to monitor the progress of this dangerous hurricane,'' the service said, adding, ``Little change in strength is expected for the next several hours as the hurricane moves westward over Jamaica''. Forecasters said the hurricane had been gaining strength as it passed over the ocean after it dumped 5 to 10 inches of rain on the Dominican Republic and Haiti, which share the island of Hispaniola. Heavy rain and stiff winds downed power lines and caused flooding in the Dominican Republi